<a href="https://colab.research.google.com/github/frankettheofranckettheo/Advanced-ML-Project/blob/tp5/tp5_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PART 1.2: Exercise 1 - Custom Attention Layer

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class SimpleAttention(layers.Layer):
    def __init__(self, **kwargs):
        super(SimpleAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape est sous la forme (batch_size, seq_len, hidden_dim)

        # W : Matrice de poids pour calculer le score d'importance
        self.W = self.add_weight(name="att_weight",
                                 shape=(input_shape[-1], 1),
                                 initializer="normal")

        # b : Biais (spécifique à chaque pas de temps selon la consigne shape=(input_shape[1], 1))
        self.b = self.add_weight(name="att_bias",
                                 shape=(input_shape[1], 1),
                                 initializer="zeros")

        super(SimpleAttention, self).build(input_shape)

    def call(self, x):
        # x est la sortie du GRU : (batch_size, seq_len, hidden_dim)

        # ---------------------------------------------------------
        # TODO 1: Implement the score calculation (dot product + tanh)
        # ---------------------------------------------------------
        # Calcul : e = tanh(W * x + b)
        # x . W va donner une forme (batch_size, seq_len, 1)
        e = tf.tanh(tf.matmul(x, self.W) + self.b)

        # ---------------------------------------------------------
        # TODO 2: Apply Softmax to get the alignment weights
        # ---------------------------------------------------------
        # On applique le softmax sur l'axe du temps (axis=1) pour que la somme des poids soit 1
        a = tf.nn.softmax(e, axis=1)

        # ---------------------------------------------------------
        # TODO 3: Compute the context vector (weighted sum of x)
        # ---------------------------------------------------------
        # On multiplie chaque vecteur caché (x) par son poids d'attention (a)
        # x : (batch, seq, hidden) * a : (batch, seq, 1) -> Broadcasting
        weighted_input = x * a

        # On somme sur l'axe du temps pour obtenir un seul vecteur par exemple
        context_vector = tf.reduce_sum(weighted_input, axis=1)

        return context_vector, a

# ==========================================
# Construction du modèle (Dernier TODO du PDF)
# ==========================================

def build_model_with_attention(seq_len, input_dim, hidden_dim):
    # 1. Input Layer
    inputs = keras.Input(shape=(seq_len, input_dim))

    # 2. GRU Layer (Doit retourner les séquences pour que l'attention puisse choisir)
    # return_sequences=True est OBLIGATOIRE ici
    gru_out = layers.GRU(hidden_dim, return_sequences=True)(inputs)

    # 3. Custom SimpleAttention Layer
    # Elle prend toute la séquence du GRU et retourne un context vector
    context_vector, attention_weights = SimpleAttention()(gru_out)

    # 4. Dense Layer (Classification ou Régression finale)
    outputs = layers.Dense(1, activation='linear')(context_vector)

    model = keras.Model(inputs=inputs, outputs=outputs, name="GRU_Attention_Model")
    return model

# ==========================================
# Test d'exécution
# ==========================================
if __name__ == "__main__":
    # Paramètres arbitraires pour le test
    SEQ_LEN = 20
    INPUT_DIM = 5
    HIDDEN_DIM = 64

    # Création du modèle
    model = build_model_with_attention(SEQ_LEN, INPUT_DIM, HIDDEN_DIM)

    # Affichage du résumé pour vérifier les connexions
    model.summary()

    # Petit test avec des données aléatoires
    dummy_input = tf.random.normal((32, SEQ_LEN, INPUT_DIM))
    output = model(dummy_input)

    print("\nTest réussi : Forme de la sortie finale =", output.shape)
    # Doit afficher (32, 1) car batch_size=32 et Dense=1

Model: "GRU_Attention_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20, 5)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 20, 64)         │        13,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_attention_1              │ [(None, 64), (None,    │            84 │
│ (SimpleAttention)               │ 20, 1)]                │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,781 (53.83 KB)

 Trainable params: 13,781 (53.83 KB)

 Non-trainable params: 0 (0.00 B)


Test réussi : Forme de la sortie finale = (32, 1)


# PART 2.1: Exercise 2 - Hybrid LSTM-Attention

In [5]:
!pip install tensorflow mlflow matplotlib numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.8 MB/s eta 0:00:00


In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import mlflow
import mlflow.tensorflow
import io

# ==========================================
# 1. Génération du Dataset Synthétique
# ==========================================
def generate_synthetic_data(n_samples=1000, seq_len=50, output_len=20):
    """
    Génère des séries temporelles (somme de sinus) + bruit.
    Input (X): seq_len pas de temps.
    Output (y): output_len pas de temps futurs (prédiction).
    """
    X = []
    y = []
    t_input = np.linspace(0, 50, seq_len)
    t_output = np.linspace(50, 70, output_len)

    for _ in range(n_samples):
        # Paramètres aléatoires pour la complexité
        freq1 = np.random.uniform(0.1, 0.5)
        freq2 = np.random.uniform(0.05, 0.2)
        phase = np.random.uniform(0, 2*np.pi)

        # Fonction génératrice
        def wave_func(t):
            return np.sin(freq1 * t + phase) + 0.5 * np.sin(freq2 * t)

        signal_in = wave_func(t_input)
        noise = np.random.normal(0, 0.05, seq_len)

        signal_out = wave_func(t_output) # On prédit le futur propre (sans bruit)

        X.append(signal_in + noise)
        y.append(signal_out)

    # Reshape pour RNN : (Samples, Timesteps, Features)
    return np.array(X)[..., np.newaxis], np.array(y)[..., np.newaxis]

# ==========================================
# 2. Construction du Modèle Hybride
# ==========================================
def build_hybrid_model(input_seq_len, output_seq_len, feature_dim=1):

    # --- ENCODEUR (Bi-Directional LSTM) ---
    enc_inputs = keras.Input(shape=(input_seq_len, feature_dim), name="enc_input")

    # return_sequences=True : nécessaire pour que l'attention accède à tous les pas de temps
    # return_state=True : nécessaire pour initialiser le décodeur
    lstm_enc = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, return_state=True),
        name="bi_lstm_enc"
    )
    enc_out, fwd_h, fwd_c, bwd_h, bwd_c = lstm_enc(enc_inputs)

    # Fusion des états (Forward + Backward) pour initialiser le décodeur
    state_h = layers.Concatenate()([fwd_h, bwd_h])
    state_c = layers.Concatenate()([fwd_c, bwd_c])
    encoder_states = [state_h, state_c] # Dimension 128

    # --- DÉCODEUR (avec Teacher Forcing) ---
    dec_inputs = keras.Input(shape=(output_seq_len, feature_dim), name="dec_input")

    # Le LSTM Décodeur doit avoir dim=128 car c'est la concaténation du Bi-LSTM (64*2)
    dec_lstm = layers.LSTM(128, return_sequences=True, return_state=True, name="dec_lstm")

    # On initialise le décodeur avec les états finaux de l'encodeur
    dec_out, _, _ = dec_lstm(dec_inputs, initial_state=encoder_states)

    # --- CROSS-ATTENTION ---
    # Query (Q) : Sortie du Décodeur (ce qu'on cherche à prédire)
    # Value (V) / Key (K) : Sortie de l'Encodeur (le contexte source)
    # return_attention_scores=True nous permet de visualiser les poids plus tard
    attention_layer = layers.Attention(name="cross_attention")

    # Note: Keras Attention prend [query, value] (ou [query, value, key])
    context_vector, attn_scores = attention_layer(
        [dec_out, enc_out],
        return_attention_scores=True
    )

    # Concaténation Contexte + Sortie LSTM Décodeur
    concat = layers.Concatenate(axis=-1)([dec_out, context_vector])

    # Tête de prédiction finale
    outputs = layers.TimeDistributed(layers.Dense(feature_dim))(concat)

    # On définit le modèle pour renvoyer aussi les scores d'attention pour l'analyse
    model = keras.Model(inputs=[enc_inputs, dec_inputs], outputs=[outputs, attn_scores])

    # Pour l'entrainement standard, on veut souvent masquer les scores dans la loss,
    # mais ici on va garder une sortie double et ajuster la loss weights.
    return model

# ==========================================
# 3. MLOps: Callback pour tracker l'Attention
# ==========================================
class AttentionLogger(keras.callbacks.Callback):
    def __init__(self, val_data, log_freq=1):
        super(AttentionLogger, self).__init__()
        self.val_data = val_data # (X_val, dec_val, y_val)
        self.log_freq = log_freq

    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.log_freq == 0:
            # 1. Faire une prédiction sur un exemple de validation
            X_sample, dec_sample, y_sample = self.val_data
            X_sample = X_sample[:1] # Prendre le premier du batch
            dec_sample = dec_sample[:1]

            # Pred -> [predictions, attention_scores]
            preds, attn_scores = self.model.predict([X_sample, dec_sample], verbose=0)

            # attn_scores shape: (1, output_len, input_len)
            attn_map = attn_scores[0]

            # 2. Créer la Heatmap avec Matplotlib
            fig, ax = plt.subplots(figsize=(8, 6))
            cax = ax.matshow(attn_map, cmap='viridis')
            fig.colorbar(cax)
            ax.set_xlabel('Encoder Timesteps (Input Context)')
            ax.set_ylabel('Decoder Timesteps (Prediction)')
            ax.set_title(f'Attention Map - Epoch {epoch}')

            # 3. Sauvegarder l'image en mémoire et logger dans MLflow
            buf = io.BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)
            plt.close(fig)

            # MLflow Logging
            mlflow.log_image(key=f"attention_map_epoch_{epoch}", image=np.array(plt.imread(buf, format='png')))

            # Calculer une métrique simple "Attention Span"
            # (Ecart type moyen des poids : si bas = focus précis, si haut = diffus)
            attn_std = np.mean(np.std(attn_map, axis=1))
            mlflow.log_metric("avg_attention_spread", attn_std, step=epoch)
            print(f" [MLflow] Logged Attention Map & Spread for epoch {epoch}")

# ==========================================
# 4. Exécution Principale
# ==========================================
if __name__ == "__main__":
    # Paramètres
    SEQ_LEN_IN = 50
    SEQ_LEN_OUT = 20
    BATCH_SIZE = 32
    EPOCHS = 10

    # Données
    X, y = generate_synthetic_data(n_samples=1000, seq_len=SEQ_LEN_IN, output_len=SEQ_LEN_OUT)

    # Préparation du "Decoder Input" pour le Teacher Forcing
    # C'est simplement la cible décalée de 1 (ou un vecteur initial de zéros)
    # Pour simplifier ici : Input Décodeur = vecteur de zéros (Inférence pure) ou y décalé.
    # Utilisons une approche "Forcing" simple : Entrée = y (avec un token start artificiel 0)
    decoder_input_data = np.zeros_like(y)
    decoder_input_data[:, 1:, :] = y[:, :-1, :] # Shift time right

    # Split Train/Val
    split = 800
    X_train, dec_train, y_train = X[:split], decoder_input_data[:split], y[:split]
    X_val, dec_val, y_val = X[split:], decoder_input_data[split:], y[split:]

    # Configuration MLflow
    mlflow.set_experiment("TP5_Hybrid_RNN_Attention")

    with mlflow.start_run():
        # Log Params
        mlflow.log_param("encoder_type", "Bi-LSTM")
        mlflow.log_param("seq_len_in", SEQ_LEN_IN)

        # Build Model
        model = build_hybrid_model(SEQ_LEN_IN, SEQ_LEN_OUT)

        # Compile
        # La sortie du modèle est [predictions, attention_scores].
        # On veut optimiser 'predictions' (MSE) mais on se fiche de l'erreur sur 'attention_scores'.
        model.compile(
            optimizer='adam',
            loss=['mse', None], # 'None' ignore la loss pour la 2eme sortie (scores)
            loss_weights=[1.0, 0.0] # Poids 0 sur les scores
        )

        model.summary()

        # Custom Callback
        attn_logger = AttentionLogger(val_data=(X_val, dec_val, y_val))

        # Train
        history = model.fit(
            [X_train, dec_train], [y_train, np.zeros((len(y_train), SEQ_LEN_OUT, SEQ_LEN_IN))],
            validation_data=([X_val, dec_val], [y_val, np.zeros((len(y_val), SEQ_LEN_OUT, SEQ_LEN_IN))]),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[attn_logger]
        )

        # Log final metrics
        final_loss = history.history['loss'][-1]
        mlflow.log_metric("final_mse", final_loss)

        print("Training Complete. Check MLflow UI for Attention Maps.")

2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/01/26 07:26:18 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/01/26 07:26:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/26 07:26:19 INFO mlflow.store.db.utils: Updating database tables
2026/01/26 07:26:19 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/26 07:26:19 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/26 07:26:19 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/01/26 07:2

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_input           │ (None, 50, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bi_lstm_enc         │ [(None, 50, 128), │     33,792 │ enc_input[0][0]   │
│ (Bidirectional)     │ (None, 64),       │            │                   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input           │ (None, 20, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ bi_lstm_enc[0][1… │
│ (Concatenate)       │                   │            │ bi_lstm_enc[0][3] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128)       │          0 │ bi_lstm_enc[0][2… │
│ (Concatenate)       │                   │            │ bi_lstm_enc[0][4] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm (LSTM)     │ [(None, 20, 128), │     66,560 │ dec_input[0][0],  │
│                     │ (None, 128),      │            │ concatenate[0][0… │
│                     │ (None, 128)]      │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cross_attention     │ [(None, 20, 128), │          0 │ dec_lstm[0][0],   │
│ (Attention)         │ (None, 20, 50)]   │            │ bi_lstm_enc[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 20, 256)   │          0 │ dec_lstm[0][0],   │
│ (Concatenate)       │                   │            │ cross_attention[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 20, 1)     │        257 │ concatenate_2[0]… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 100,609 (393.00 KB)

 Trainable params: 100,609 (393.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.4885 [MLflow] Logged Attention Map & Spread for epoch 0
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - loss: 0.4858 - val_loss: 0.2647
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.2041 [MLflow] Logged Attention Map & Spread for epoch 1
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 0.2023 - val_loss: 0.0687
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0613 [MLflow] Logged Attention Map & Spread for epoch 2
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - loss: 0.0608 - val_loss: 0.0318
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0275 [MLflow] Logged Attention Map & Spread for epoch 3
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.0273 - val_loss: 0.0155
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 0.0144 [MLflow] Logged Attention Map & Spread for epoch 4
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - loss: 0.0143 - val_loss: 0.0115
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 58